# TomatoCare Phase 4 — frozen UniRepLKNet-A + DINOv3
Research notebook only; the deployed baseline API/PWA is unchanged.

1. Enable **Internet** and **GPU T4** in Kaggle settings.
2. Attach `thanhhainguyn/tomatocare-tomato-leaf-disease-4-classes`.
3. Add a Kaggle secret named **HF_TOKEN**, enable notebook access, and use the HF account already granted DINOv3 access.
4. Source must contain the reviewed Phase 4 files. The setup clones the research branch `feature/hybrid-dinov3`. Alternatively attach a source-code input and set `REPO_DIR` accordingly; do not use production main for Phase 4.

Both pretrained encoders stay frozen/eval. The default run performs a real-pretrained smoke test only. Training and final held-out test require separate explicit opt-ins below. Do not print the token or publish checkpoints containing DINOv3 weights without reviewing its license.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/TomatoCare-hybrid")  # Or the attached Phase 4 source checkout.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "feature/hybrid-dinov3", "--single-branch", "https://github.com/NguyenThanhHaii/TomatoCare.git", str(REPO_DIR)], check=True)
if not (REPO_DIR / "tomatocare/hybrid_model.py").is_file():
    raise RuntimeError("Phase 4 source is missing. Use a fresh directory cloned from feature/hybrid-dinov3 or attach that branch checkout.")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-hybrid.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))
import torch
assert torch.cuda.is_available(), "Enable Kaggle T4 GPU before loading encoders"
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from tomatocare.train_hybrid import HybridTrainConfig, validate_hybrid_bundle, make_loader, seed_everything

config = HybridTrainConfig(batch_size=4, epochs=5, num_workers=0, device="cuda")
seed_everything(config.seed)
bundle = validate_hybrid_bundle(config.data_root)
print("Dataset:", bundle.root)
print("Class order:", bundle.class_names)
print("Counts:", {split: len(rows) for split, rows in bundle.records.items()})


In [ ]:
from kaggle_secrets import UserSecretsClient
from tomatocare.hybrid_model import create_hybrid_model, processor_configuration

# Do not login globally, print credentials, or put them in config/checkpoint fields.
try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Enable notebook access to Kaggle secret HF_TOKEN") from None
try:
    model, processor = create_hybrid_model(hf_token=token)
finally:
    del token
print("Encoder provenance:", model.provenance)
print("DINOv3 preprocessing:", processor_configuration(processor))


## Real-pretrained smoke gate
Expected: Stage 3 GAP `[B,160]`, DINOv3 CLS `[B,384]`, concatenation `[B,544]`, logits `[B,4]`. CLS is token 0; the following register tokens are not pooled. DINOv3 pixels come from its own downloaded processor, independently of the baseline RGB/bilinear 224×224 ImageNet transform. The same optional horizontal flip is applied to both branches during training only.

Batch size 4 is a conservative T4 starting point, not a measured guarantee. Use 2 or 1 if this gate reports CUDA out-of-memory, then record the final settings before training. No automatic fallback to random encoders is allowed.


In [ ]:
from tomatocare.train_hybrid import smoke_test

loader = make_loader(bundle.records["train"], processor, config, train=False)
batch = next(iter(loader))
torch.cuda.reset_peak_memory_stats()
smoke = smoke_test(model, batch, config.device)
print("Real-pretrained smoke:", smoke)
print("GPU peak allocated MiB:", torch.cuda.max_memory_allocated() / 2**20)
assert smoke["unireplknet"] == [len(batch[1]), 160]
assert smoke["dinov3_cls"] == [len(batch[1]), 384]
assert smoke["fusion"] == [len(batch[1]), 544]
assert smoke["logits"] == [len(batch[1]), 4]
assert smoke["trainable_parameters"] == 3268


## Optional training — disabled by default
Use validation Macro-F1 for selection. Freeze both encoders and their BatchNorm statistics; only LayerNorm(544) + Linear(544,4) train. Encoders use `torch.no_grad()` and the fusion head keeps gradients. Default: five epochs, batch 4, AdamW LR 0.001 / weight decay 0.05, seed 42.

The saved `.pth` is explicitly **head-only**: it excludes DINOv3 and UniRepLKNet encoder weights, retaining source revisions/hashes and preprocessing for reconstruction. Keep outputs private pending license review. Training never evaluates test automatically.


In [ ]:
from tomatocare.train_hybrid import run_hybrid_training

RUN_TRAINING = False  # Explicitly enable only after reviewing the smoke gate.
best_path = None
if RUN_TRAINING:
    best_path = run_hybrid_training(config, model, processor)
    print("Best head checkpoint:", best_path)
else:
    print("Training disabled; architecture/pretrained smoke only.")


## Optional final test — disabled separately
Lock hyperparameters using validation first. Use the held-out 820-image test only after model selection; never tune against its results. Read saved metrics on subsequent visits instead of reevaluating. Compare baseline and hybrid on identical splits, class mapping and evaluation definitions; report that the hybrid has different pretraining and extra compute, not only a changed classifier.


In [ ]:
from tomatocare.train_hybrid import evaluate_final_test

RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if best_path is None:
        raise RuntimeError("Select a completed validation-best head checkpoint first")
    metrics = evaluate_final_test(config, model, processor, best_path)
    print(metrics)
else:
    print("Held-out test remains untouched.")


## Outputs, limitations and license
Download private outputs from `/kaggle/working/tomatocare_hybrid/`: best head `.pth`, `history.json`, validation metrics/confusion-matrix CSV/PNG, and optional one-time test outputs. Recreate the two encoders from saved provenance (use resolved revisions, and verify baseline SHA-256) before restoring the head; authorized DINOv3 access is still required. No HF token is included in these artifacts.

Exact SHA-256 duplicate images may repeat within a split/class. Splits are not verified independent by `leaf_id`, and near-duplicates may inflate metrics. Do not fabricate a hybrid improvement over the reported baseline 91.46% Accuracy / 90.51% Macro-F1.

DINOv3 uses Meta's custom [DINOv3 License](https://github.com/facebookresearch/dinov3/blob/main/LICENSE.md), not Apache/MIT. Distribution of DINO Materials/derivatives is subject to its terms and requires a copy of the agreement; research publications must acknowledge use. Source discussion and a license copy are in `docs/hybrid-phase4.md` and `docs/third-party/DINOv3-LICENSE.md`. No full DINOv3-containing hybrid checkpoint is automatically exported or uploaded. Recheck the gated model's current accepted terms before any public release.
